# End of week 1 exercise
Use selenium stealth to scrape a few stock related webpage for top news then summarize for insights

In [1]:
import sys
import time
from typing import List, Union

from bs4 import BeautifulSoup 
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from loguru import logger
from openai import OpenAI 
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium_stealth import stealth
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
logger.remove()
logger.add(sys.stderr, level="INFO")
MODEL_GPT = 'gpt-4.1-nano'
MODEL_OLLAMA = 'qwen2.5vl:latest'
FINANCE_WEBSITES = ["https://finance.yahoo.com/", "https://www.reuters.com/markets/", "https://marketwatch.com"]
website_information_cache = {}

In [3]:
load_dotenv(override=True)
openai_client = OpenAI() 
ollama_client = OpenAI(
    base_url = "http://localhost:11434/v1",
    api_key="ollama"
)

In [4]:
class Website:
    def __init__(self, url:str, wait_time:int) -> None:
        """
        Create this Website object from the given url using the BeautifulSoup library and selenium
        """
        self.url = url
        self.wait_time = wait_time
        self.scrape_data()
        self.extract_page_info()

    def scrape_data(self):
        options = webdriver.ChromeOptions()
        options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("start-maximized")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("useAutomationExtension", False)
    
        try:
            logger.info("setting up chrome webdriver...")
            service = Service(ChromeDriverManager().install())
            driver = webdriver.Chrome(service=service, options=options) 
        except Exception as exc:
            logger.error(f"Error setting up webdrver: {exc}") 
            return None
    
        stealth(
            driver, 
            languages=["en-US", "en"],
            vendor="Google Inc.",
            platform="Win32",
            webgl_vendor="Intel Inc.",
            renderer="Intel Iris OpenGL Engine",
            fix_hairline=True,
        )
        logger.info("Selenium Stealth has been applied.")
    
        logger.info(f"Navigating to {self.url}")
        driver.get(self.url)
    
        try:
            logger.info("waiting for page elements to load..")
            wait = WebDriverWait(driver, self.wait_time)
            wait.until(EC.presence_of_all_elements_located((By.TAG_NAME, "body")))
            logger.info("Page Body is found.")
            time.sleep(3)
        except Exception as exc:
            logger.error(f"Error waiting for page elements: {exc}")
            driver.quit()
            return None
        self.html_source = driver.page_source

    def extract_page_info(self) -> None:
        logger.info("extracting page information")
        soup = BeautifulSoup(self.html_source, 'html.parser')
        logger.info("extracting title: ...")
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        logger.info("extracting page text: ...")
        self.text = soup.body.get_text(separator="\n", strip=True)

In [5]:
system_prompt = """
You are an trained investment analyst and you are great at synthesizing daily news from various websites. 
Given the contents of a list of websites, extract key information from each website content. 
Ingore text that might be navigation related and combine duplicated materials. 
Respond in markdown. 
"""

In [6]:
def get_user_prompt(websites: List[str]) -> str:
    user_prompt = "You are looking at the following list of websites with the following title and text information:\n"
    for url in websites:
        if url not in website_information_cache:
            website_info = Website(url, 30)
            website_information_cache[url] = website_info
        title, text = website_information_cache[url].title, website_information_cache[url].text
        user_prompt += f"website title: {title}\n"
        user_prompt += f"website text : {text}\n"
        user_prompt += "\n\n"
    user_prompt += "Please summarize the key information and create a list of action items."
    return user_prompt 

In [7]:
def get_insights(websites: List[str], model_client, model_name) -> None:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": get_user_prompt(websites)}
    ]
    stream = model_client.chat.completions.create(model=model_name, messages = messages, stream=True)
    response = ""
    display_handler = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        response = response.replace("```", "").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handler.display_id)

## Use Ollama and "qwen" 

In [9]:
get_insights(FINANCE_WEBSITES, ollama_client, MODEL_OLLAMA)

2025-08-03 15:10:11.717 | INFO     | __main__:scrape_data:21 - setting up chrome webdriver...
2025-08-03 15:10:13.517 | INFO     | __main__:scrape_data:37 - Selenium Stealth has been applied.
2025-08-03 15:10:13.518 | INFO     | __main__:scrape_data:39 - Navigating to https://finance.yahoo.com/
2025-08-03 15:10:14.938 | INFO     | __main__:scrape_data:43 - waiting for page elements to load..
2025-08-03 15:10:15.071 | INFO     | __main__:scrape_data:46 - Page Body is found.
2025-08-03 15:10:18.154 | INFO     | __main__:extract_page_info:55 - extracting page information
2025-08-03 15:10:20.380 | INFO     | __main__:extract_page_info:57 - extracting title: ...
2025-08-03 15:10:20.388 | INFO     | __main__:extract_page_info:61 - extracting page text: ...
2025-08-03 15:10:20.391 | INFO     | __main__:scrape_data:21 - setting up chrome webdriver...
2025-08-03 15:10:22.119 | INFO     | __main__:scrape_data:37 - Selenium Stealth has been applied.
2025-08-03 15:10:22.120 | INFO     | __main__:s

### Summary of Key Information

The article discusses the broader economic and market implications stemming from the ongoing global trade disputes, particularly focusing on tariffs announced by the United States under President Trump. This includes:

- **Economic Context:** The article highlights that the US economy is showing signs of potential slowing growth due to these disputes, and how it could impact other major economies globally.
- **Market Impact:** There is scrutiny over the effects on stock markets, where the US stock markets (S&P 500) have shown some volatility and mixed performance during 2018, with an overall decline of around 6% in 2019.
- **Fed's Reaction:** The article notes how the Federal Reserve's meeting this week could reflect the impact of the trade disputes on monetary policy decisions. 
- **Stock Market and Investment Concerns:** There is discussion about individual stocks and sectors, such as Nasdaq and others, showing signs of distress, partly due to trade tensions.
- **Consumer and Investment Trends:** Financial news and investment advice sections are provided, focusing on strategies for retirement planning, stock market investing, and managing personal financial goals for both short and long term.

---

### Action Items

1. **Track Market Index Performance:**
   - Monitor the performance of major indices such as S&P 500, Nasdaq, and Dow Jones as trade tensions evolve to determine their impact on global markets and economic recovery.

2. **Adjust Portfolio Management:**
   - Revisit individual stock and portfolio selections in light of current market volatility and potential losses on sectors affected by trade tariffs.

3. **Evaluate Consumer Spending and Inflation Expectations:**
   - Assess how increasing tariffs might affect consumer spending and inflation trends by tracking GDP growth and CPI changes.

4. **Evaluate Federal Reserve's Policy:**
   - Keep an eye on the Federal Reserve’s next policy meeting to anticipate any adjustments in monetary policy likely to be influenced by the trade disputes.

5. **Stay Informed on Trade Deal Progress:**
   - Monitor ongoing trade negotiations and see if any significant diplomatic breakthroughs can mitigate the trade disputes' economic impact.

6. **Consult with Financial Advisors:**
   - Reach out to financial planners or investment banks specializing in geopolitical investments to understand a more detailed strategy to navigate economic fluctuations during trade wars.

7. **Stay Updated on Investment Advice and Financial Reports:**
   - Follow regular stock analysis reports and financial advice columns in reputable sources for insights into which sectors might be best insulated during such economic challenges.
  
8. **Educate Yourself on Tax Strategies:**
   - Be aware of the tax implications of current and future tariff policies, and how to manage income tax during times of economic distress.

9. **Plan Retirement Safeguardly:**
   - Ensure your retirement savings strategy is prepared for economic downturns and consider investing in inflation-protected assets or low-risk mutual funds.

By focusing on these action items, one can be better prepared to navigate through the economic implications of trade disputes.

## Use OpenAI and gpt 4.1 nano

In [10]:
get_insights(FINANCE_WEBSITES, openai_client, MODEL_GPT)

# Market and Economic Summary - Key Highlights

## Stock Market Overview
- U.S. major indices declined significantly today:
  - Dow Jones: -1.23% (43,588.58)
  - S&P 500: -1.60% (6,238.01)
  - Nasdaq: -2.24% (20,650.13)
- Global markets are also down, with European and Asian indices posting declines (e.g., Euro STOXX 50 -2.90%, Nikkei -0.66%).

## Market Drivers & Sentiment
- Concerns over economic growth persist amid geopolitical tensions, tariffs, and weak job data.
- Weak July jobs report: US added only 73,000 jobs — far below expectations; unemployment up to 4.2%. Revisions point to softer labor conditions.
- Bond and dollar markets signal recession fears; U.S. 10-year Treasury yield at 4.23%, indicating investor caution.
- Stock volatility remains high, with the VIX index rising over 21%.

## Key Market Events & Trends
- Significant tariff policies continue to influence markets:
  - Trump has increased tariffs on Canada (to 35%) and threatened with tariffs on other countries.
  - Trump has also sought to pressure pharma companies to reduce drug prices.
  - Trump recently granted Mexico a tariff reprieve for 90 days.
  - Pending trade deals, including with South Korea, which involves a 15% tariff.
- Corporate earnings season shows mixed signals:
  - Over 100 S&P 500 companies report upcoming earnings, including Palantir, Eli Lilly, and Disney.
  - Amazon reports earnings but stock drops on guidance concerns; Apple beats expectations with strong iPhone sales.
  - Nvidia and Microsoft see huge gains driven by AI and cloud revenues.
- Notable earnings and corporate activity:
  - Nvidia's market cap surpasses $4 trillion.
  - Alphabet's CapitalG is in talks to fund Vast Data at up to $30 billion valuation.
  - Figma IPO success suggests increased IPO activity in software sector.

## Geopolitical & Global Market News
- OPEC+ has announced a substantial oil output increase of 547,000 barrels/day, reversing previous cuts.
- Swiss officials may revise their position on Trump tariffs.
- Brazil’s Azul airline secures $650 million investment.
- UK’s FCA proposes a large auto-finance redress scheme ($12-24 billion).

## Policy & Central Bank Outlook
- Federal Reserve left interest rates unchanged but remains divided:
  - Two dissenters advocate for rate cuts due to economic uncertainties.
  - Powell highlights inflation concerns with a recent rise in prices.
- U.S. inflation measures show increases, partly attributed to tariffs and global factors.
- ECB and BOE consider or prepare for potential rate adjustments amidst inflation pressures.

## Other Notable Market Trends
- The 'great wealth transfer' is ongoing, with many individuals unprepared.
- The US housing market remains volatile, with mortgage rates and credit rules under scrutiny.
- Cryptocurrencies (e.g., ETH) remain volatile but with signs of bullish activity.

---

# Action Items & Recommendations

1. **Re-evaluate Portfolio Risks**:
   - Prepare for increased volatility due to geopolitical tensions, tariffs, and weak economic data.
   - Consider reducing exposure to high-volatility sectors (e.g., tech, biotech) if risk appetite diminishes.

2. **Monitor Inflation & Federal Reserve Policy**:
   - Stay alert to inflation trends and Fed statements; potential pause or pivot in rate policy could impact markets.

3. **Watch for Corporate Earnings & Guidance**:
   - Pay attention to key reportings, especially technology giants and industrials, for market sentiment.
   - Assess corporate resilience amid global tensions and tariff impacts.

4. **Evaluate Impact of Tariff Policies**:
   - Consider geopolitical developments affecting supply chains, commodity prices, and international trade.

5. **Prepare for Sector Shifts**:
   - Energy (OPEC) and commodities may experience volatility; adjust positions accordingly.
   - Technology and AI sectors remain attractive but could face correction if broader market declines continue.

6. **Global and Geopolitical Risk Management**:
   - Keep an eye on developments in the Middle East, China, and Europe that could influence markets.

7. **Stay Agile with Real-time Data**:
   - Use timely tools and analytics (e.g., real-time market data, earnings reports) to adjust strategies proactively.

---

*Note:* Given current volatility and macroeconomic uncertainties, maintaining a diversified and cautious approach is advisable. Further analysis of sector-specific fundamentals and geopolitical risks is recommended before making substantial moves.